# HUMEA main-table reproduction on Kaggle

This notebook reuses the authors' released multimodal features and trains HUMEA itself. Start with `MODE = "smoke"`, then run `single`, and use `all` only after validation succeeds.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, zipfile

MODE = "single"       # validate | smoke | single | all
EXPERIMENT = "db15k-20"
BATCH_SIZE = 384          # Paper: 512; T4 x2 has 15 GiB per GPU
DOWNLOAD_DATA_IF_MISSING = True
WORK_ROOT = Path("/kaggle/working/HUMEA")
INPUT_ROOT = Path("/kaggle/input")
DATA_URL = "https://drive.google.com/file/d/1a3aou1qe7Yzq6y_khyTq1La_kf3UI7sC/view?usp=drive_link"
assert MODE in {"validate", "smoke", "single", "all"}
print("Mode:", MODE, "Experiment:", EXPERIMENT, "Batch size:", BATCH_SIZE)

In [ ]:
import torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Visible GPUs:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(index, torch.cuda.get_device_name(index))
subprocess.run(["nvidia-smi"], check=False)

## Load the Kaggle-ready source

Attach the private Kaggle Dataset created from `HUMEA-kaggle-bundle.zip`. The cell automatically finds `kaggle_runner.py` and copies the source into writable storage.

In [ ]:
runner_candidates = list(INPUT_ROOT.rglob("kaggle_runner.py"))
if not runner_candidates:
    source_archives = list(INPUT_ROOT.rglob("HUMEA-kaggle-bundle.zip"))
    if source_archives:
        source_extract = Path("/kaggle/working/HUMEA-source")
        source_extract.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(source_archives[0]) as source_zip:
            source_zip.extractall(source_extract)
        runner_candidates = list(source_extract.rglob("kaggle_runner.py"))
if not runner_candidates:
    raise FileNotFoundError(
        "Attach the private Kaggle Dataset made from HUMEA-kaggle-bundle.zip; "
        "kaggle_runner.py was not found under /kaggle/input."
    )
source_root = runner_candidates[0].parent
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
shutil.copytree(
    source_root, WORK_ROOT,
    ignore=shutil.ignore_patterns(".git", ".worktrees", "__pycache__", "artifacts"),
)
print("Copied source from", source_root, "to", WORK_ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "loguru", "gdown"], check=True)

## Locate or download the processed features

Attached, already-extracted `mmkb-datasets` is preferred. Otherwise the notebook extracts an attached `data.zip`; with Internet enabled it can download the authors' archive once.

In [ ]:
expected = WORK_ROOT / "data" / "mmkb-datasets"
if not expected.is_dir():
    extracted_candidates = [p for p in INPUT_ROOT.rglob("mmkb-datasets") if p.is_dir()]
    if extracted_candidates:
        data_link = WORK_ROOT / "data"
        if data_link.exists() or data_link.is_symlink():
            if data_link.is_dir() and not data_link.is_symlink():
                shutil.rmtree(data_link)
            else:
                data_link.unlink()
        data_link.symlink_to(extracted_candidates[0].parent, target_is_directory=True)
        print("Using attached extracted data:", extracted_candidates[0])
    else:
        archives = list(INPUT_ROOT.rglob("data.zip"))
        archive = archives[0] if archives else Path("/kaggle/working/humea-data.zip")
        if not archives:
            if not DOWNLOAD_DATA_IF_MISSING:
                raise FileNotFoundError("No attached data.zip and downloading is disabled")
            subprocess.run(["gdown", "--fuzzy", DATA_URL, "-O", str(archive)], check=True)
        (WORK_ROOT / "data").mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as data_zip:
            data_zip.extractall(WORK_ROOT / "data")
        print("Extracted", archive)
assert expected.is_dir(), f"Expected dataset directory was not created: {expected}"
print("Dataset root:", expected)

In [ ]:
base_command = [
    sys.executable, "kaggle_runner.py", "--repo-root", str(WORK_ROOT),
    "--batch-size", str(BATCH_SIZE),
]
validate = subprocess.run(base_command + ["validate"], cwd=WORK_ROOT)
if validate.returncode != 0:
    raise RuntimeError("Preflight validation failed; review the missing-file output above")

In [ ]:
if MODE == "validate":
    run_command = None
elif MODE == "smoke":
    run_command = base_command + ["smoke"]
elif MODE == "single":
    run_command = base_command + ["single", "--experiment", EXPERIMENT]
else:
    run_command = base_command + ["all"]
if run_command:
    print("Running:", " ".join(run_command))
    completed = subprocess.run(run_command, cwd=WORK_ROOT)
    if completed.returncode != 0:
        raise RuntimeError(f"HUMEA runner failed with exit code {completed.returncode}")

In [ ]:
import pandas as pd
summary_path = WORK_ROOT / "artifacts" / "summary.csv"
if summary_path.is_file():
    display(pd.read_csv(summary_path))
    archive_path = shutil.make_archive(
        "/kaggle/working/HUMEA-results", "zip", WORK_ROOT, "artifacts"
    )
    print("Saved results:", archive_path)
else:
    print("Validation finished; no training summary was created.")